# utils


In [37]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pathlib import Path
from scipy import optimize, stats
from statsmodels.tsa.stattools import adfuller, coint
import vectorbtpro as vbt


In [ ]:
def fit_price_spread(log_price_frame, left_symbol, right_symbol):
    pair_log = log_price_frame[[left_symbol, right_symbol]].dropna()
    y, x = pair_log[left_symbol], pair_log[right_symbol]

    # y = alpha + beta*x + spread
    xmat = np.column_stack([np.ones(len(pair_log)), x])
    alpha, beta = np.linalg.lstsq(xmat, y, rcond=None)[0]
    spread = y - alpha - beta * x

    return alpha, beta, spread


def ar1_half_life(spread):
    ar_frame = pd.concat(
        [
            spread.diff().rename("delta"),
            spread.shift(1).rename("lag"),
        ],
        axis=1,
    ).dropna()

    ar_x = np.column_stack([np.ones(len(ar_frame)), ar_frame["lag"]])
    ar_phi = np.linalg.lstsq(ar_x, ar_frame["delta"], rcond=None)[0][1]

    level_phi = 1.0 + ar_phi
    if 0.0 < level_phi < 1.0:
        theta = -np.log(level_phi)
        return np.log(2.0) / theta, level_phi, theta

    return np.inf, level_phi, np.nan


In [ ]:
def ou_transition_mean(x0, dt, speed, level):
    return level + (x0 - level) * np.exp(-speed * dt)


def ou_transition_std(dt, speed, vola):
    return np.sqrt(vola * vola * (1.0 - np.exp(-2.0 * speed * dt)) / (2.0 * speed))


def ou_loglik(times, values, speed, level, vola):
    dt = np.diff(times)
    mu = ou_transition_mean(values[:-1], dt, speed, level)
    sigma = ou_transition_std(dt, speed, vola)
    return np.sum(stats.norm.logpdf(values[1:], loc=mu, scale=sigma))


def fit_ou_mle(series):
    clean = series.replace([np.inf, -np.inf], np.nan).dropna().astype(float)
    values = clean.to_numpy()
    times = np.arange(len(values), dtype=float)

    y = values[1:]
    z = values[:-1]
    xmat = np.column_stack([np.ones(len(z)), z])
    c_hat, a_hat = np.linalg.lstsq(xmat, y, rcond=None)[0]
    residual = y - c_hat - a_hat * z

    if 0.0 < a_hat < 1.0:
        speed_start = -np.log(a_hat)
        level_start = c_hat / (1.0 - a_hat)
        q_hat = max(np.mean(residual * residual), 1e-12)
        vola_start = np.sqrt(q_hat * 2.0 * speed_start / max(1.0 - a_hat * a_hat, 1e-12))
    else:
        speed_start = 0.05
        level_start = float(np.mean(values))
        vola_start = float(np.std(np.diff(values), ddof=1))

    start = np.array(
        [
            max(speed_start, 1e-6),
            level_start,
            max(vola_start, 1e-8),
        ],
        dtype=float,
    )

    def objective(theta):
        speed, level, vola = theta
        if speed <= 0 or vola <= 0:
            return np.inf
        ll = ou_loglik(times, values, speed, level, vola)
        return -ll if np.isfinite(ll) else np.inf

    result = optimize.minimize(
        objective,
        start,
        method="L-BFGS-B",
        bounds=[(1e-8, None), (None, None), (1e-10, None)],
        options={"maxiter": 500},
    )

    speed, level, vola = result.x if result.success else start

    return {
        "speed": float(speed),
        "level": float(level),
        "vola": float(vola),
        "half_life_bars": float(np.log(2.0) / speed),
        "stationary_std": float(vola / np.sqrt(2.0 * speed)),
        "loglik": float(ou_loglik(times, values, speed, level, vola)),
        "success": bool(result.success),
        "n_obs": int(len(values)),
    }


In [ ]:
fee_bps = 4
slippage_bps = 2
exit_score = 0.0
max_holding_bars = 24 * 14


def make_signal_path(score, entry, exit_level, stop_level, max_hold, score_col="score"):
    current_position = 0
    bars_held = 0
    records = []

    for time, value in score.dropna().items():
        value = float(value)
        prior_position = current_position

        if current_position == 0:
            bars_held = 0
            if value <= -entry:
                current_position = 1
            elif value >= entry:
                current_position = -1

        else:
            bars_held += 1
            exit_signal = (current_position > 0 and value >= -exit_level) or (current_position < 0 and value <= exit_level)
            stop_signal = (current_position > 0 and value <= -stop_level) or (current_position < 0 and value >= stop_level)
            time_stop = bars_held >= max_hold

            if exit_signal:
                current_position = 0
                bars_held = 0
            elif stop_signal:
                current_position = 0
                bars_held = 0
            elif time_stop:
                current_position = 0
                bars_held = 0

        records.append(
            {
                "timestamp": time,
                score_col: value,
                "entry_score": entry,
                "exit_score": exit_level,
                "stop_score": stop_level,
                "prior_position": prior_position,
                "position": current_position,
                "bars_held": bars_held,
            }
        )

    return pd.DataFrame(records).set_index("timestamp")


In [ ]:
def run_pair_backtest(two_leg_price, hedge, position):
    gross_exposure = 1 + abs(hedge)
    weights = pd.DataFrame(
        {
            two_leg_price.columns[0]: position / gross_exposure,
            two_leg_price.columns[1]: -position * hedge / gross_exposure,
        },
        index=position.index,
    ).reindex(two_leg_price.index).fillna(0.0)

    # signal o close t, execute o bar tiep theo
    execution_weights = weights.shift(1).fillna(0.0)

    pf = vbt.Portfolio.from_orders(
        close=two_leg_price[[two_leg_price.columns[0], two_leg_price.columns[1]]],
        size=execution_weights,
        size_type="targetpercent",
        fees=fee_bps / 10_000,
        slippage=slippage_bps / 10_000,
        init_cash=1.0,
        cash_sharing=True,
        group_by=True,
        direction="both",
    )

    vbt_stats = pf.stats()
    summarize = {
        "net_return": vbt_stats["Total Return [%]"],
        "sharpe": vbt_stats["Sharpe Ratio"],
        "max_drawdown": vbt_stats["Max Drawdown [%]"],
        "position_coverage": vbt_stats["Position Coverage [%]"],
        "total_trades": vbt_stats["Total Trades"],
        "win_rate": vbt_stats["Win Rate [%]"],
        "profit_factor": vbt_stats["Profit Factor"],
        "total_fees_paid": vbt_stats["Total Fees Paid"],
    }

    return pf, summarize, weights


# data


In [ ]:
frames = []

data_dir = Path(r"D:/quant idea local/quant-idea/crypto_pairs_trading/stat_arb/csv/csv")

for path in sorted(data_dir.glob("PERP_*_USDT_1h.csv")):
    frame = pd.read_csv(path, usecols=["bar_time", "symbol", "close", "volume"])
    frame["bar_time"] = pd.to_datetime(frame["bar_time"])
    frame[["close", "volume"]] = frame[["close", "volume"]].apply(pd.to_numeric, errors="coerce")
    frame["quote_volume"] = frame["close"] * frame["volume"]
    frames.append(frame)

raw = pd.concat(frames, ignore_index=True)
raw.head()


,bar_time,symbol,close,volume,quote_volume
0,2022-01-01 00:00:00,PERP:1000SHIB/USDT,0.033823,522172816.0,1.766145e+07
1,2022-01-01 01:00:00,PERP:1000SHIB/USDT,0.033799,483119781.0,1.632897e+07
2,2022-01-01 02:00:00,PERP:1000SHIB/USDT,0.033741,242157420.0,8.170634e+06
3,2022-01-01 03:00:00,PERP:1000SHIB/USDT,0.033639,239812063.0,8.067038e+06
4,2022-01-01 04:00:00,PERP:1000SHIB/USDT,0.033457,334288861.0,1.118430e+07


In [ ]:
prices = raw.pivot(index="bar_time", columns="symbol", values="close").sort_index()
quote_volume = raw.pivot(index="bar_time", columns="symbol", values="quote_volume").sort_index()

prices = prices.loc[prices.index.intersection(quote_volume.index)]
quote_volume = quote_volume.loc[prices.index]

print(prices.shape)
prices.tail()


(38665, 90)


symbol,PERP:1000SHIB/USDT,PERP:1000XEC/USDT,PERP:1INCH/USDT,PERP:AAVE/USDT,PERP:ADA/USDT,PERP:ALGO/USDT,PERP:ALICE/USDT,PERP:ANKR/USDT,PERP:AR/USDT,PERP:ARPA/USDT,PERP:ATOM/USDT,PERP:AVAX/USDT,PERP:AXS/USDT,PERP:BAND/USDT,PERP:BAT/USDT,PERP:BCH/USDT,PERP:BEL/USDT,PERP:BNB/USDT,PERP:BTC/USDT,PERP:BTCDOM/USDT,PERP:C98/USDT,PERP:CELO/USDT,PERP:CELR/USDT,PERP:CHR/USDT,PERP:CHZ/USDT,PERP:COMP/USDT,PERP:COTI/USDT,PERP:CRV/USDT,PERP:CTSI/USDT,PERP:DASH/USDT,PERP:DOGE/USDT,PERP:DOT/USDT,PERP:DYDX/USDT,PERP:EGLD/USDT,PERP:ENJ/USDT,PERP:ENS/USDT,PERP:ETC/USDT,PERP:ETH/USDT,PERP:FIL/USDT,PERP:GALA/USDT,PERP:GRT/USDT,PERP:GTC/USDT,PERP:HBAR/USDT,PERP:HOT/USDT,PERP:ICX/USDT,PERP:IOST/USDT,PERP:IOTA/USDT,PERP:IOTX/USDT,PERP:KAVA/USDT,PERP:KNC/USDT,PERP:KSM/USDT,PERP:LINK/USDT,PERP:LPT/USDT,PERP:LTC/USDT,PERP:MANA/USDT,PERP:MASK/USDT,PERP:MTL/USDT,PERP:NEAR/USDT,PERP:NEO/USDT,PERP:OGN/USDT,PERP:ONE/USDT,PERP:ONT/USDT,PERP:PEOPLE/USDT,PERP:QTUM/USDT,PERP:RLC/USDT,PERP:ROSE/USDT,PERP:RSR/USDT,PERP:RUNE/USDT,PERP:RVN/USDT,PERP:SAND/USDT,PERP:SFP/USDT,PERP:SKL/USDT,PERP:SNX/USDT,PERP:SOL/USDT,PERP:STORJ/USDT,PERP:SUSHI/USDT,PERP:THETA/USDT,PERP:TRB/USDT,PERP:TRX/USDT,PERP:UNI/USDT,PERP:VET/USDT,PERP:XLM/USDT,PERP:XMR/USDT,PERP:XRP/USDT,PERP:XTZ/USDT,PERP:YFI/USDT,PERP:ZEC/USDT,PERP:ZEN/USDT,PERP:ZIL/USDT,PERP:ZRX/USDT
bar_time,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2026-05-30 20:00:00,0.005532,0.00695,0.0873,83.34,0.2374,0.1276,0.1244,0.004601,2.414,0.01022,2.028,8.995,1.210,0.2020,0.1183,306.52,0.1043,719.88,73944.5,5465.0,0.01973,0.07671,0.002600,0.02145,0.03337,18.41,0.01218,0.2166,0.02885,40.19,0.10125,1.197,0.1889,3.592,0.03906,6.019,8.241,2026.30,0.985,0.003134,0.02636,0.09520,0.09664,0.000376,0.03816,0.001034,0.0637,0.00445,0.0563,0.1403,4.566,9.253,2.121,52.51,0.0852,0.4376,0.2883,2.312,2.709,0.02148,0.001844,0.0526,0.00640,0.870,0.4539,0.00927,0.001694,0.4336,0.00526,0.07005,0.3030,0.00584,0.2986,82.91,0.0934,0.1958,0.1875,16.781,0.34721,3.070,0.006135,0.24174,380.84,1.3473,0.3292,2298.0,534.39,5.779,0.00380,0.1035
2026-05-30 21:00:00,0.005532,0.00696,0.0873,83.21,0.2374,0.1262,0.1245,0.004588,2.387,0.01020,2.027,8.994,1.211,0.2019,0.1196,305.47,0.1036,719.60,73954.8,5459.7,0.01972,0.07632,0.002600,0.02129,0.03307,18.49,0.01222,0.2163,0.02882,39.88,0.10126,1.198,0.1851,3.596,0.03902,6.015,8.243,2027.54,0.984,0.003127,0.02671,0.09519,0.09641,0.000375,0.03821,0.001034,0.0638,0.00447,0.0562,0.1402,4.572,9.259,2.126,52.55,0.0848,0.4365,0.2875,2.295,2.711,0.02142,0.001835,0.0526,0.00641,0.869,0.4540,0.00920,0.001688,0.4314,0.00527,0.07003,0.3032,0.00583,0.2974,82.96,0.0928,0.1957,0.1877,16.746,0.34737,3.072,0.006161,0.24020,370.38,1.3495,0.3319,2301.0,531.77,5.757,0.00381,0.1034
2026-05-30 22:00:00,0.005497,0.00691,0.0867,82.70,0.2359,0.1247,0.1236,0.004554,2.384,0.01012,2.011,8.937,1.193,0.2001,0.1186,302.45,0.1032,716.27,73849.8,5478.8,0.01957,0.07581,0.002575,0.02124,0.03277,18.29,0.01212,0.2151,0.02875,39.45,0.10064,1.187,0.1854,3.562,0.03877,5.972,8.199,2022.51,0.969,0.003099,0.02636,0.09435,0.09525,0.000372,0.03804,0.001028,0.0629,0.00439,0.0560,0.1390,4.540,9.188,2.109,52.38,0.0846,0.4340,0.2862,2.267,2.703,0.02133,0.001824,0.0522,0.00636,0.862,0.4469,0.00912,0.001677,0.4281,0.00522,0.06939,0.3054,0.00577,0.2962,82.61,0.0925,0.1939,0.1855,16.614,0.34745,3.047,0.006119,0.22887,369.65,1.3419,0.3293,2293.0,526.12,5.721,0.00379,0.1026
2026-05-30 23:00:00,0.005494,0.00690,0.0868,82.74,0.2358,0.1245,0.1234,0.004563,2.403,0.01011,2.017,8.942,1.193,0.2008,0.1196,303.48,0.1029,719.01,73856.8,5471.7,0.01963,0.07577,0.002578,0.02130,0.03284,18.27,0.01212,0.2147,0.02873,39.41,0.10056,1.189,0.1854,3.569,0.03866,5.972,8.209,2021.58,0.972,0.003105,0.02644,0.09345,0.09510,0.000372,0.03802,0.001027,0.0621,0.00441,0.0562,0.1393,4.552,9.187,2.108,52.43,0.0846,0.4339,0.2860,2.248,2.706,0.02138,0.001827,0.0521,0.00636,0.864,0.4474,0.00912,0.001679,0.4278,0.00522,0.06940,0.3047,0.00577,0.2960,82.69,0.0917,

In [ ]:
idx = np.arange(len(prices))

train_frac = 0.70
formation_frac = 0.40

research_start = int(idx[0])
research_end = int(idx[-1] + 1)

formation_start = research_start
train_end = research_start + int((research_end - research_start) * train_frac)
formation_end = research_start + int((train_end - research_start) * formation_frac)
validation_start = formation_end
validation_end = train_end
test_start = validation_end
test_end = research_end

# DOT/FIL


In [ ]:
symbol_a = "PERP:DOT/USDT"
symbol_b = "PERP:FIL/USDT"

formation_price = prices.iloc[formation_start:formation_end][[symbol_a, symbol_b]].dropna()
formation_log_price = np.log(formation_price)

alpha, beta, formation_spread = fit_price_spread(formation_log_price, symbol_a, symbol_b)

coin_stat, coin_pvalue, _ = coint(formation_log_price[symbol_a], formation_log_price[symbol_b])
adf_pvalue = adfuller(formation_spread.dropna(), autolag="AIC")[1]
half_life_bars, level_phi, theta = ar1_half_life(formation_spread)

pair_context = pd.DataFrame(
    [
        {
            "symbol_a": symbol_a,
            "symbol_b": symbol_b,
            "alpha": alpha,
            "beta": beta,
            "coin_pvalue": coin_pvalue,
            "adf_pvalue": adf_pvalue,
            "ar1_level_phi": level_phi,
            "half_life_bars": half_life_bars,
        }
    ]
)

pair_context


,symbol_a,symbol_b,alpha,beta,coin_pvalue,adf_pvalue,ar1_level_phi,half_life_bars
0,PERP:DOT/USDT,PERP:FIL/USDT,0.542228,0.794317,0.00185,0.00031,0.99572,161.605975


# validation calibration


In [ ]:
entry_grid = [1.25, 1.50, 1.75, 2.00, 2.25, 2.50]
stop_mult_grid = [1.5, 2.0, 2.5, 3.0]
rolling_window_grid = [72, 120, 168, 336]


def build_scores(fit_start, fit_end, score_start, score_end, A, B):
    
    fit_price = prices.iloc[fit_start:fit_end][[A, B]].dropna()
    score_price = prices.iloc[score_start:score_end][[A, B]].dropna()

    alpha, beta, fit_spread = fit_price_spread(np.log(fit_price), A, B)
    score_log = np.log(score_price)
    spread = score_log[A] - alpha - beta * score_log[B]

    ou_fit = fit_ou_mle(fit_spread)

    # OU z-score 
    ou_score = ((spread - ou_fit["level"]) / ou_fit["stationary_std"]).rename("ou_score")

    rolling_scores = {}
    # rolling std z-score
    for zscore_window in rolling_window_grid:
        rolling_scores[zscore_window] = (
            (spread - spread.rolling(zscore_window).mean()) / spread.rolling(zscore_window).std()
        ).rename("rolling_zscore")

    return {
        "alpha": alpha,
        "beta": beta,
        "spread": spread,
        "ou_fit": ou_fit,
        "ou_score": ou_score,
        "rolling_scores": rolling_scores,
    }


In [ ]:
validation_model = build_scores(
    formation_start,
    formation_end,
    formation_start,
    validation_end,
    symbol_a,
    symbol_b,
)

validation_price = prices.iloc[validation_start:validation_end][[symbol_a, symbol_b]].dropna()

thres_rows = []

for entry in entry_grid:
    for stop_mult in stop_mult_grid:
        stop = max(stop_mult * entry, entry + 0.5)
        ou_hold = int(np.clip(np.ceil(3.0 * validation_model["ou_fit"]["half_life_bars"]), 24, max_holding_bars))
        val_score = validation_model["ou_score"].loc[validation_price.index].dropna()
        signal = make_signal_path(val_score, entry, exit_score, stop, ou_hold, score_col="ou_score")
        _, bt_stats, _ = run_pair_backtest(validation_price, validation_model["beta"], signal["position"])

        thres_rows.append(
            {
                "strategy": "ou_score",
                "entry_score": entry,
                "zscore_window": np.nan,
                "stop_mult": stop_mult,
                "stop_score": stop,
                "validation_return": bt_stats["net_return"],
                "validation_sharpe": bt_stats["sharpe"],
                "validation_max_drawdown": bt_stats["max_drawdown"],
                "validation_entry_count": int(((signal["position"].shift(1).fillna(0) == 0) & (signal["position"] != 0)).sum()),
                "validation_active_bar_ratio": float((signal["position"] != 0).mean()),
            }
        )

        for zscore_window, rolling_score in validation_model["rolling_scores"].items():
            val_score = rolling_score.loc[validation_price.index].dropna()
            signal_path = make_signal_path(val_score, entry, exit_score, stop, max_holding_bars, score_col="rolling_zscore")
            _, bt_stats, _ = run_pair_backtest(validation_price, validation_model["beta"], signal_path["position"])

            thres_rows.append(
                {
                    "strategy": "rolling_zscore",
                    "entry_score": entry,
                    "zscore_window": zscore_window,
                    "stop_mult": stop_mult,
                    "stop_score": stop,
                    "validation_return": bt_stats["net_return"],
                    "validation_sharpe": bt_stats["sharpe"],
                    "validation_max_drawdown": bt_stats["max_drawdown"],
                    "validation_entry_count": int(((signal_path["position"].shift(1).fillna(0) == 0) & (signal_path["position"] != 0)).sum()),
                    "validation_active_bar_ratio": float((signal_path["position"] != 0).mean()),
                }
            )

thres_table = pd.DataFrame(thres_rows)
valid_thres_table = thres_table[thres_table["validation_entry_count"] >= 5].copy()

ou_params = (
    valid_thres_table[valid_thres_table["strategy"].eq("ou_score")]
    .sort_values(["validation_return", "validation_sharpe"], ascending=False)
    .head(1)
)

rolling_params = (
    valid_thres_table[valid_thres_table["strategy"].eq("rolling_zscore")]
    .sort_values(["validation_return", "validation_sharpe"], ascending=False)
    .head(1)
)

selected_params = pd.concat([rolling_params, ou_params])
selected_params


           strategy  entry_score  zscore_window  stop_mult  stop_score  validation_return  validation_sharpe  validation_max_drawdown  validation_entry_count  \
113  rolling_zscore          2.5          168.0        2.5        6.25          46.618199           1.052647                19.840049                      93   
35         ou_score          1.5            NaN        3.0        4.50          99.034881           1.777881                13.937224                      26   

     validation_active_bar_ratio  
113                     0.500400  
35                      0.470411  


# rolling std - baseline


In [ ]:
test_model = build_scores(
    formation_start,
    validation_end,
    formation_start,
    test_end,
    symbol_a,
    symbol_b,
)

test_price = prices.iloc[test_start:test_end][[symbol_a, symbol_b]].dropna()

rolling_selected = selected_params[selected_params["strategy"].eq("rolling_zscore")].iloc[0]
rolling_entry_score = rolling_selected["entry_score"]
rolling_stop_score = rolling_selected["stop_score"]
rolling_zwin = rolling_selected["zscore_window"]

test_rolling_score = test_model["rolling_scores"][rolling_zwin].loc[test_price.index].dropna()
signal_path = make_signal_path(
    test_rolling_score,
    rolling_entry_score,
    exit_score,
    rolling_stop_score,
    max_holding_bars,
    score_col="rolling_zscore",
)

part2_pf, part2_stats, part2_weights = run_pair_backtest(test_price, test_model["beta"], signal_path["position"])

part2_summary = pd.DataFrame(
    [
        {
            "sample": "test",
            "strategy": "rolling_zscore_baseline",
            "entry_score": rolling_entry_score,
            "zscore_window": rolling_zwin,
            "stop_score": rolling_stop_score,
            **part2_stats,
        }
    ]
)

part2_summary


,sample,strategy,entry_score,zscore_window,stop_score,net_return,sharpe,max_drawdown,position_coverage,total_trades,win_rate,profit_factor,total_fees_paid
0,test,rolling_zscore_baseline,2.5,168.0,6.25,13.527999,0.554151,26.612695,40.793103,2607,43.109405,1.07891,0.058033


# OU


In [ ]:
ou_selected = selected_params[selected_params["strategy"].eq("ou_score")].iloc[0]
ou_entry_score = ou_selected["entry_score"]
ou_stop_score = ou_selected["stop_score"]

ou_fit = test_model["ou_fit"]
ou_hold = np.clip(np.ceil(3 * ou_fit["half_life_bars"]), 24, max_holding_bars)

test_ou_score = test_model["ou_score"].loc[test_price.index].dropna()
signal_path1 = make_signal_path(
    test_ou_score,
    ou_entry_score,
    exit_score,
    ou_stop_score,
    ou_hold,
    score_col="ou_score",
)

part3_pf, part3_stats, part3_weights = run_pair_backtest(test_price, test_model["beta"], signal_path1["position"])

part3_summary = pd.DataFrame(
    [
        {
            "sample": "test",
            "strategy": "ou_score",
            "entry_score": ou_entry_score,
            "zscore_window": np.nan,
            "stop_score": ou_stop_score,
            "ou_half_life_bars": ou_fit["half_life_bars"],
            "ou_stationary_std": ou_fit["stationary_std"],
            **part3_stats,
        }
    ]
)

test_summary = pd.concat([part2_summary, part3_summary], ignore_index=True)
test_summary


,sample,strategy,entry_score,zscore_window,stop_score,net_return,sharpe,max_drawdown,position_coverage,total_trades,win_rate,profit_factor,total_fees_paid,ou_half_life_bars,ou_stationary_std
0,test,rolling_zscore_baseline,2.5,168.0,6.25,13.527999,0.554151,26.612695,40.793103,2607,43.109405,1.078910,0.058033,NaN,NaN
1,test,ou_score,1.5,NaN,4.50,24.598386,0.853872,14.821119,59.620690,3070,42.438070,1.214418,0.028980,252.072772,0.088461
